# Hi-res 1601² CFDAC — CNN zoo (shallow + deep + cnn3d) (GPU)

All three CFDAC CNNs in one notebook — **`cnn2d_shallow`** (light, 128-baseline architecture), **`cnn2d_deep`** (`DeepCFDACNet`, ResNet18-style, consumes the full 1601² grid), and **`cnn3d`** (channels as a 3-D depth axis) — on full-1601² CFDAC × 7 features × 10 tasks (3×7×10 = 210 cells). One shared `cnn` results folder/branch with skip-if-exists + pickup, so it reuses anything already trained. `BATCH=16` is sized for the heavy `cnn2d_deep`/`cnn3d` at 1601² — raise it for shallow-only runs.

**No GPU? Set Runtime → Change runtime type → GPU.** Private repos: add a Colab secret `GH_TOKEN`. Results persist to Google Drive; **File → Save a copy in GitHub** saves this notebook itself. Set `CELLS` to one tuple to run a single cell per session.

## 1 · Bootstrap (clone phd_lanl + pymodal, install deps incl. pint/pyFRF/audiomentations)

In [1]:
import os, sys, subprocess
GH_USER='grcarmenaty'; WORK='/content'; os.chdir(WORK)
def _tok():
    try:
        from google.colab import userdata; return userdata.get('GH_TOKEN')
    except Exception: return os.environ.get('GH_TOKEN')
def clone(repo, branch, dst):
    if os.path.isdir(dst): print('exists', dst); return
    t=_tok(); auth=f'{t}@' if t else ''
    url=f'https://{auth}github.com/{GH_USER}/{repo}.git'
    assert subprocess.run(['git','clone','--depth','1','-b',branch,url,dst]).returncode==0, \
        f'clone failed {repo}@{branch} (private? add a GH_TOKEN Colab secret)'
clone('phd_lanl','main','/content/PhD_LANL')
clone('pymodal','master','/content/pymodal')   # sibling dir the scripts expect
for p in ('/content/PhD_LANL','/content/pymodal'):
    if p not in sys.path: sys.path.insert(0,p)
os.chdir('/content/PhD_LANL')
# Harden git's HTTP transport against Drive-mounted-Colab flakiness (the 408s):
for _k,_v in [('http.postBuffer','524288000'),('http.version','HTTP/1.1'),
              ('http.lowSpeedLimit','1000'),('http.lowSpeedTime','300')]:
    subprocess.run(['git','config','--global',_k,_v])
subprocess.run([sys.executable,'-m','pip','-q','install','timm','h5py','scikit-learn','pint','pyFRF','audiomentations'])
import torch
print('torch',torch.__version__,'| cuda',torch.cuda.is_available(),'|',
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU - set a GPU runtime!')

torch 2.11.0+cu128 | cuda True | NVIDIA A100-SXM4-40GB


## 2 · Regenerate the 1601-bin features (gitignored; rebuilt from committed sources)

In [2]:
import subprocess, sys, os, glob, json, h5py, numpy as np
from pathlib import Path
REPO=Path(os.getcwd())
def run(cmd): print('>>',' '.join(cmd)); assert subprocess.run(cmd).returncode==0, cmd
if not (REPO/'dataset'/'features_hires.h5').exists():
    run([sys.executable,'ml_pipeline/generate_dataset.py','--out','dataset_hires','--n-t','4096','--fs','256'])
    run([sys.executable,'ml_pipeline/build_hires_synth_features.py'])
if not (REPO/'experimental_frfs.h5').exists():
    with open('experimental_frfs.h5','wb') as o:
        for p in sorted(glob.glob('experimental_frfs_chunks/experimental_frfs.h5.part_*')):
            o.write(open(p,'rb').read())
if not (REPO/'dataset'/'experimental_features.h5').exists():
    from ml_pipeline.evaluate import primary_op
    with h5py.File('experimental_frfs.h5','r') as f: names=json.loads(f.attrs['case_names_json'])
    n=len(names); tc=np.zeros(n,np.int8); st=np.full(n,-1,np.int8); en=np.full(n,-1,np.int8); sv=np.zeros(n,np.float32)
    for i,nm in enumerate(names):
        op=primary_op(nm); tc[i]=op['type_code']; st[i]=op['storey']; en[i]=op['end']; sv[i]=op['severity']
    dt=h5py.string_dtype('utf-8')
    with h5py.File('dataset/experimental_features.h5','w') as o:
        o.create_dataset('names',data=np.array(names,dtype=object),dtype=dt)
        o.create_dataset('type_code',data=tc); o.create_dataset('storey',data=st)
        o.create_dataset('end',data=en); o.create_dataset('severity',data=sv)
if not (REPO/'dataset'/'experimental_features_hires.h5').exists():
    run([sys.executable,'ml_pipeline/build_hires_exp_features.py'])
print('features ready')

>> /usr/bin/python3 ml_pipeline/generate_dataset.py --out dataset_hires --n-t 4096 --fs 256
>> /usr/bin/python3 ml_pipeline/build_hires_synth_features.py
>> /usr/bin/python3 ml_pipeline/build_hires_exp_features.py
features ready


## 3 · Config + context (edit the CONFIG block)

In [3]:
import torch, numpy as np, h5py
from pathlib import Path
from ml_pipeline import hires_zoo as Z
from ml_pipeline.tasks import build_targets
from ml_pipeline.train import make_split
DEV=torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ===================== CONFIG (edit me) =====================
MODELS   = ['cnn2d_shallow','cnn2d_deep','cnn3d']
TASKS    = ['binary','col_location','mass_location','severity','type',
            'is_bolt','is_crack','is_mass','is_hole','is_pristine']
FEATURES = list(Z.CFDAC_FEATURES)          # all 7 CFDAC channel-features
MAX_EPOCHS = 80        # safety cap; training stops early at convergence
PATIENCE   = 8         # early-stop after this many epochs with no val gain
SUBSAMPLE= 4000        # A100 fits this easily; raise toward 10000 for more data
BATCH    = 16          # tuned to THIS model's memory footprint; drop on smaller GPUs / if OOM
VISION_SIZE = 384      # conv vision backbones feed size (swin/vit fixed 224); A100 can do 448-512
# --- GitHub autosave (each finished cell -> a per-family results branch) ---
FAMILY            = 'cnn'
AUTOSAVE_GITHUB   = True                               # set False to disable
GH_RESULTS_BRANCH = 'colab-hires-cnn'         # never touches main
# Full grid below. To run ONE cell this session set e.g.:
#   CELLS = [('type','transformer','cfdac_realimag')]
CELLS = Z.all_cfdac_cells(MODELS, TASKS, FEATURES)
print(len(CELLS),'cells queued across', MODELS)
# ============================================================

# Persistence: Google Drive survives Colab session resets (skip-if-exists resumes).
try:
    from google.colab import drive; drive.mount('/content/drive')
    OUT=Path('/content/drive/MyDrive/hires_cfdac/cnn')
except Exception:
    OUT=Path('results_hires_zoo_cnn')
OUT.mkdir(parents=True, exist_ok=True); print('OUT =', OUT)

# Pick up cells already trained in a past run: seed OUT/per_case from the
# results branch (so we never retrain past work, even on a fresh session/Drive).
_BR = 'colab-hires-cnn'
import subprocess as _sp, os as _os
try:
    _sp.run(['git','-C','/content/PhD_LANL','fetch','--depth','1','origin',_BR], capture_output=True)
    _ls = _sp.run(['git','-C','/content/PhD_LANL','ls-tree','-r','--name-only','origin/'+_BR],
                  capture_output=True, text=True).stdout
    (OUT/'per_case').mkdir(parents=True, exist_ok=True); _n=0
    for _l in _ls.splitlines():
        if 'results_hires_zoo/cnn/per_case/' in _l and _l.endswith('.json'):
            _name=_os.path.basename(_l)
            if not (OUT/'per_case'/_name).exists():
                _b=_sp.run(['git','-C','/content/PhD_LANL','show','origin/'+_BR+':'+_l],
                           capture_output=True, text=True).stdout
                if _b: (OUT/'per_case'/_name).write_text(_b); _n+=1
    print('picked up',_n,'already-trained cells from',_BR)
except Exception as _e:
    print('branch pickup skipped:', _e)

SYN=Path('dataset/features_hires.h5'); EXP=Path('dataset/experimental_features_hires.h5')
with h5py.File(SYN,'r') as f:
    syn_tasks=build_targets(f['type_code'][:].astype('int64'),f['storey'][:].astype('int64'),
                            f['end'][:].astype('int64'),f['severity'][:].astype('float32'))
    H_ref_syn=torch.from_numpy(f['reference/frf_complex'][:].astype('complex64')).to(DEV)
with h5py.File(EXP,'r') as f:
    exp_tasks=build_targets(f['type_code'][:].astype('int64'),f['storey'][:].astype('int64'),
                            f['end'][:].astype('int64'),f['severity'][:].astype('float32'))
    exp_names=[str(s) for s in f['names'][:]]
    H_ref_exp=torch.from_numpy(f['reference/frf_complex'][:].astype('complex64')).to(DEV)
with h5py.File(EXP,'r') as f:
    H_exp=(f['frf_real'][:]+1j*f['frf_imag'][:]).astype('complex64')
print('context ready; exp FRFs', H_exp.shape)
print('device', DEV, '| amp dtype', Z._amp_dtype(DEV),
      '| (bfloat16 expected on A100/H100)')

210 cells queued across ['cnn2d_shallow', 'cnn2d_deep', 'cnn3d']
Mounted at /content/drive
OUT = /content/drive/MyDrive/hires_cfdac/cnn
picked up 0 already-trained cells from colab-hires-cnn
context ready; exp FRFs (2638, 1601, 9)
device cuda | amp dtype torch.bfloat16 | (bfloat16 expected on A100/H100)


## 4 · Run the cell grid (skip-if-exists; resumes from Drive)

In [4]:
import torch, os, shutil, subprocess
def _tok():
    try:
        from google.colab import userdata; return userdata.get('GH_TOKEN')
    except Exception: return os.environ.get('GH_TOKEN')
GH_TOKEN = _tok()
if AUTOSAVE_GITHUB and not GH_TOKEN:
    print('AUTOSAVE_GITHUB is on but no GH_TOKEN secret found -> results go to Drive/zip only.')

def git_autosave(msg):
    """Force-push the JSON results of THIS family to its own results branch.
    Only per_case/*.json + synth_test_zoo.json are pushed (NOT the model
    .ckpt/.pt weights, which stay on Drive). Per-family branch => never
    conflicts with main or other families; always the full accumulated state."""
    if not (AUTOSAVE_GITHUB and GH_TOKEN): return
    repo='/content/PhD_LANL'; dst=os.path.join(repo,'results_hires_zoo',FAMILY)
    os.makedirs(os.path.join(dst,'per_case'), exist_ok=True)
    # copy ONLY the json artefacts (skip the large models/ dir)
    for fn in os.listdir(os.path.join(OUT,'per_case')) if os.path.isdir(os.path.join(OUT,'per_case')) else []:
        if fn.endswith('.json'): shutil.copy(os.path.join(OUT,'per_case',fn), os.path.join(dst,'per_case',fn))
    if os.path.exists(os.path.join(OUT,'synth_test_zoo.json')):
        shutil.copy(os.path.join(OUT,'synth_test_zoo.json'), os.path.join(dst,'synth_test_zoo.json'))
    cwd=os.getcwd(); os.chdir(repo)
    subprocess.run(['git','config','user.email','colab@gpu.run'])
    subprocess.run(['git','config','user.name','colab-gpu'])
    subprocess.run(['git','add','-f',f'results_hires_zoo/{FAMILY}/per_case',f'results_hires_zoo/{FAMILY}/synth_test_zoo.json'])
    if subprocess.run(['git','diff','--cached','--quiet']).returncode!=0:
        subprocess.run(['git','commit','-q','-m',msg])
        url=f'https://{GH_TOKEN}@github.com/grcarmenaty/phd_lanl.git'
        import time as _t; ok=False
        for _a in range(5):                       # retry the flaky push w/ backoff
            r=subprocess.run(['git','push','--force',url,f'HEAD:{GH_RESULTS_BRANCH}'],
                             capture_output=True,text=True)
            if r.returncode==0: ok=True; break
            _t.sleep(4*(2**_a))                    # 4,8,16,32,64s
        print('  autosave:', f'pushed -> {GH_RESULTS_BRANCH}' if ok
              else 'push failed after retries (results safe on Drive): '+r.stderr[-140:])
    os.chdir(cwd)

for (task, model, feature) in CELLS:
    try:
        Z.run_cell(task, model, feature, syn_h5=SYN, exp_h5=EXP, out_dir=OUT, dev=DEV,
                   syn_tasks=syn_tasks, exp_tasks=exp_tasks, H_ref_syn=H_ref_syn,
                   H_ref_exp=H_ref_exp, H_exp=H_exp, exp_names=exp_names,
                   make_split=make_split, subsample=SUBSAMPLE, batch=BATCH,
                   vision_size=VISION_SIZE, max_epochs=MAX_EPOCHS, patience=PATIENCE)
        git_autosave(f'colab autosave [{FAMILY}]: {task}/{model}/{feature}')
    except Exception as e:
        print('CELL FAILED', task, model, feature, '::', repr(e)[:200])
        if torch.cuda.is_available(): torch.cuda.empty_cache()
print('\nqueue done')

skip binary_cnn2d_shallow_cfdac_real_hires1601 (exists)
  autosave: pushed -> colab-hires-cnn
skip binary_cnn2d_shallow_cfdac_imag_hires1601 (exists)
skip binary_cnn2d_shallow_cfdac_mag_hires1601 (exists)
skip binary_cnn2d_shallow_cfdac_phase_hires1601 (exists)
skip binary_cnn2d_shallow_cfdac_realimag_hires1601 (exists)
skip binary_cnn2d_shallow_cfdac_magphase_hires1601 (exists)
skip binary_cnn2d_shallow_cfdac_all_hires1601 (exists)
skip binary_cnn2d_deep_cfdac_real_hires1601 (exists)
skip binary_cnn2d_deep_cfdac_imag_hires1601 (exists)
skip binary_cnn2d_deep_cfdac_mag_hires1601 (exists)
skip binary_cnn2d_deep_cfdac_phase_hires1601 (exists)
skip binary_cnn2d_deep_cfdac_realimag_hires1601 (exists)
skip binary_cnn2d_deep_cfdac_magphase_hires1601 (exists)
skip binary_cnn2d_deep_cfdac_all_hires1601 (exists)
skip binary_cnn3d_cfdac_real_hires1601 (exists)
skip binary_cnn3d_cfdac_imag_hires1601 (exists)
skip binary_cnn3d_cfdac_mag_hires1601 (exists)
skip binary_cnn3d_cfdac_phase_hires1601 (e

## 5 · Honest summary (balanced-acc / macro-F1 / collapse) + zip download

In [5]:
import json, numpy as np
from pathlib import Path
from collections import Counter
from sklearn.metrics import balanced_accuracy_score, f1_score, accuracy_score
print(f"{'cell':<48}{'kind':>5}{'synth':>8}{'expMF1/R2':>11}{'expBal':>8}{'expAcc':>8}{'collapse':>9}")
print('-'*97)
for p in sorted((OUT/'per_case').glob('*_hires1601.json')):
    d=json.loads(p.read_text()); m=d['meta']; r=d['rows']
    yt=np.array([x['y_true'] for x in r]); yp=np.array([x['y_pred'] for x in r])
    name=f"{m['task']}/{m['model']}/{m['feature']}"
    if m['kind']=='cls':
        n=m['n_out']; bal=balanced_accuracy_score(yt,yp)
        mf1=f1_score(yt,yp,labels=list(range(n)),average='macro',zero_division=0); acc=accuracy_score(yt,yp)
        coll=(len(set(yp.tolist()))<=1) or (bal<=1/n+0.02)
        print(f"{name:<48}{'cls':>5}{(m.get('synth_test_macro_f1') or 0):>8.3f}{mf1:>11.3f}{bal:>8.3f}{acc:>8.3f}{str(coll):>9}")
    else:
        yt=yt.astype(float); yp=yp.astype(float); ss=np.sum((yt-yp)**2); st=np.sum((yt-yt.mean())**2)
        r2=1-ss/st if st>0 else 0; mae=np.mean(np.abs(yt-yp))
        print(f"{name:<48}{'reg':>5}{(m.get('synth_test_metric') or 0):>8.3f}{r2:>11.3f}{'-':>8}{'-':>8}{'MAE=%.3f'%mae:>9}")

# Zip for download (Drive already persists across sessions).
import shutil
z=str(OUT).rstrip('/').split('/')[-1]
shutil.make_archive('/content/'+z,'zip',str(OUT))
try:
    from google.colab import files; files.download('/content/'+z+'.zip')
except Exception as e: print('zip at /content/'+z+'.zip', e)

cell                                             kind   synth  expMF1/R2  expBal  expAcc collapse
-------------------------------------------------------------------------------------------------
binary/cnn2d_deep/cfdac_all                       cls   0.889      0.451   0.498   0.821     True
binary/cnn2d_deep/cfdac_imag                      cls   0.773      0.439   0.452   0.726     True
binary/cnn2d_deep/cfdac_mag                       cls   0.688      0.491   0.514   0.818     True
binary/cnn2d_deep/cfdac_magphase                  cls   0.780      0.475   0.486   0.755     True
binary/cnn2d_deep/cfdac_phase                     cls   0.796      0.438   0.468   0.770     True
binary/cnn2d_deep/cfdac_real                      cls   0.729      0.427   0.422   0.641     True
binary/cnn2d_deep/cfdac_realimag                  cls   0.837      0.448   0.476   0.775     True
binary/cnn2d_shallow/cfdac_all                    cls   0.599      0.297   0.492   0.298     True
binary/cnn2d_shallow

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 6 · (Optional) push JSON results back to the repo

In [6]:
# Optional: force the full JSON snapshot to the results branch now (JSON only,
# no model weights). Same robust path as the per-cell autosave; safe to re-run.
import os, subprocess, shutil, glob, time as _t
tok=None
try:
    from google.colab import userdata; tok=userdata.get('GH_TOKEN')
except Exception: tok=os.environ.get('GH_TOKEN')
if not tok:
    print('No GH_TOKEN - download the zip from the cell above and hand it to the agent.')
else:
    repo='/content/PhD_LANL'; dst=os.path.join(repo,'results_hires_zoo',FAMILY)
    os.makedirs(os.path.join(dst,'per_case'), exist_ok=True)
    for fn in (os.listdir(os.path.join(OUT,'per_case')) if os.path.isdir(os.path.join(OUT,'per_case')) else []):
        if fn.endswith('.json'): shutil.copy(os.path.join(OUT,'per_case',fn), os.path.join(dst,'per_case',fn))
    for sj in glob.glob(os.path.join(OUT,'synth_test_*.json')):
        shutil.copy(sj, os.path.join(dst, os.path.basename(sj)))
    os.chdir(repo)
    subprocess.run(['git','config','user.email','colab@gpu.run']); subprocess.run(['git','config','user.name','colab-gpu'])
    subprocess.run(['git','add','-f',f'results_hires_zoo/{FAMILY}'])
    subprocess.run(['git','commit','-q','-m',f'hires {FAMILY} (GPU): manual JSON snapshot'])
    url=f'https://{tok}@github.com/grcarmenaty/phd_lanl.git'; ok=False
    for _a in range(5):
        r=subprocess.run(['git','push','--force',url,f'HEAD:{GH_RESULTS_BRANCH}'],capture_output=True,text=True)
        if r.returncode==0: ok=True; break
        _t.sleep(4*(2**_a))
    print(f'pushed -> {GH_RESULTS_BRANCH}' if ok else 'push failed after retries: '+r.stderr[-200:])

pushed -> colab-hires-cnn
